### Esta es la version

# Chapter 2 - Lab 5: Financial News Agent with Evaluator-Optimizer Pattern

This version keeps the **Evaluator-Optimizer** pattern, but avoids Reuters domain filtering at the hosted-search level. In testing, the domain-restricted search returned zero structured sources even when matching Reuters articles existed. The agent now searches the public web broadly and the application filters retrieved/cited URLs to Reuters programmatically.


## 1. Install dependencies


In [19]:
!pip install -U openai openai-agents -q


After upgrading packages in Colab, restart the runtime if requested, then continue from the next cell.


## 2. Imports and API key


In [20]:
from collections.abc import Mapping, Sequence
from dataclasses import dataclass
from datetime import datetime, timedelta
from typing import Any, Literal
from urllib.parse import urlsplit, urlunsplit
import os
from google.colab import userdata
from agents import Agent, ItemHelpers, ModelSettings, Runner, TResponseInputItem, WebSearchTool
OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY


## 3. Helpers to inspect real web-search sources


In [21]:
@dataclass(frozen=True)
class URLCitation:
    title: str
    url: str

def get_field(obj: Any, key: str) -> Any:
    if isinstance(obj, Mapping):
        return obj.get(key)
    return getattr(obj, key, None)

def extract_url_citations(items: Sequence[Any]) -> list[URLCitation]:
    citations: list[URLCitation] = []
    seen: set[str] = set()
    for item in items:
        raw_item = get_field(item, "raw_item")
        if get_field(raw_item, "type") != "message":
            continue
        content = get_field(raw_item, "content")
        if not isinstance(content, list):
            continue
        for part in content:
            if get_field(part, "type") != "output_text":
                continue
            annotations = get_field(part, "annotations")
            if not isinstance(annotations, list):
                continue
            for annotation in annotations:
                if get_field(annotation, "type") != "url_citation":
                    continue
                url = get_field(annotation, "url")
                title = get_field(annotation, "title")
                if not isinstance(url, str) or url in seen:
                    continue
                seen.add(url)
                citations.append(URLCitation(title=title if isinstance(title, str) else url, url=url))
    return citations

def extract_web_search_source_urls(items: Sequence[Any]) -> list[str]:
    urls: list[str] = []
    seen: set[str] = set()
    for item in items:
        raw_item = get_field(item, "raw_item")
        if get_field(raw_item, "type") != "web_search_call":
            continue
        action = get_field(raw_item, "action")
        sources = get_field(action, "sources") if action else None
        if not isinstance(sources, list):
            continue
        for source in sources:
            url = get_field(source, "url")
            if not isinstance(url, str) or url in seen:
                continue
            seen.add(url)
            urls.append(url)
    return urls

def normalize_reuters_url(url: str) -> str | None:
    try:
        parsed = urlsplit(url)
    except ValueError:
        return None
    host = parsed.hostname.lower().rstrip(".") if parsed.hostname else ""
    if not (host == "reuters.com" or host.endswith(".reuters.com")):
        return None
    if parsed.scheme not in {"http", "https"}:
        return None
    path = parsed.path.rstrip("/")
    if not path:
        return None
    return urlunsplit(("https", host, path, "", ""))

def unique_reuters_urls(urls: Sequence[str]) -> list[str]:
    result: list[str] = []
    seen: set[str] = set()
    for url in urls:
        normalized = normalize_reuters_url(url)
        if normalized is None or normalized in seen:
            continue
        seen.add(normalized)
        result.append(normalized)
    return result


## 4. Searcher and evaluator agents

Hosted web search is deliberately unrestricted. Python validates Reuters URLs after retrieval. Both agents use `gpt-4.1-mini`.


In [22]:
today_date = datetime.now().strftime("%Y-%m-%d")
two_days_ago = (datetime.now() - timedelta(days=2)).strftime("%Y-%m-%d")
INSTRUCTIONS_NEWS_SEARCH = f"""
You are a financial news research agent. Find genuine Reuters articles relevant to the user's request. Use general web search; do NOT apply a Reuters-only domain filter to the search engine.
Allowed date window: {two_days_ago} through {today_date}, inclusive.
Search broadly but only Reuters articles count as final items. Use queries containing Reuters plus the requested companies/topics, and search individual companies/topics separately when necessary. Third-party pages may be discovery clues only. A valid final source must be a direct reuters.com URL. Do not use stock/ETF/index quotes as news. Do not fabricate.
Return the requested number of items (default 5). For every item include headline, publication date, short summary, publisher Reuters, and an inline web citation. If fewer verified items are found, return only those verified items.
"""
web_news_searcher = Agent(
    name="web_news_searcher",
    model="gpt-4.1-mini",
    instructions=INSTRUCTIONS_NEWS_SEARCH,
    tools=[WebSearchTool(search_context_size="high", external_web_access=True)],
    model_settings=ModelSettings(tool_choice="required", response_include=["web_search_call.action.sources"]),
)

@dataclass
class EvaluationFeedback:
    feedback: str
    score: Literal["successful", "unsuccessful"]

INSTRUCTIONS_NEWS_EVALUATOR = f"""
You are a strict evaluator of a Reuters financial-news result. You receive the original request, the news summary, Reuters URL citations, and Reuters URLs retrieved by web search.
A result is successful only if the requested number of relevant items is present (default 5), each item has headline/date/summary, dates are between {two_days_ago} and {today_date}, and enough genuine Reuters URLs support the items. Market quotes do not count as news. Do not invent requirements absent from the original request.
Return successful only if all applicable requirements are met; otherwise return unsuccessful with actionable feedback.
"""
news_evaluator = Agent(name="news_evaluator", model="gpt-4.1-mini", instructions=INSTRUCTIONS_NEWS_EVALUATOR, output_type=EvaluationFeedback)


## 5. Evaluator-Optimizer loop with retrieval diagnostics


In [23]:
async def main() -> None:
    msg = input("User's request: " ).strip()
    max_iterations = 4
    latest_outline = ""
    evaluator_feedback: str | None = None
    for iteration in range(1, max_iterations + 1):
        if evaluator_feedback is None:
            search_input: list[TResponseInputItem] = [{"content": msg, "role": "user"}]
        else:
            search_input = [{"content": msg, "role": "user"}, {"content": ("The previous attempt failed evaluation.\n\n" + f"Evaluator feedback:\n{evaluator_feedback}\n\n" + "Perform a NEW broad web search. Search individual companies/topics separately and look specifically for direct Reuters article pages. Do not reuse unsupported claims."), "role": "user"}]
        news_searcher_result = await Runner.run(web_news_searcher, search_input)
        latest_outline = ItemHelpers.text_message_outputs(news_searcher_result.new_items)
        citations = extract_url_citations(news_searcher_result.new_items)
        all_retrieved_urls = extract_web_search_source_urls(news_searcher_result.new_items)
        cited_reuters_urls = unique_reuters_urls([c.url for c in citations])
        retrieved_reuters_urls = unique_reuters_urls(all_retrieved_urls)
        print("\n\033[92m" + f"************************** NEWS SEARCH {iteration} **************************" + "\033[0m")
        print(latest_outline)
        print("\n\033[93m************************** WEB SEARCH DIAGNOSTICS **************************\033[0m")
        print(f"All URLs retrieved by web search: {len(all_retrieved_urls)}")
        for url in all_retrieved_urls[:20]: print("  SOURCE:", url)
        print(f"Reuters citations in final response: {len(cited_reuters_urls)}")
        for url in cited_reuters_urls: print("  CITED REUTERS:", url)
        print(f"Reuters URLs retrieved by web search: {len(retrieved_reuters_urls)}")
        for url in retrieved_reuters_urls: print("  RETRIEVED REUTERS:", url)
        evaluator_input = (f"ORIGINAL USER REQUEST:\n{msg}\n\nNEWS SUMMARY:\n{latest_outline}\n\n" + "REUTERS URL CITATIONS:\n" + ("\n".join(cited_reuters_urls) if cited_reuters_urls else "NONE") + "\n\nREUTERS URLS RETRIEVED BY WEB SEARCH:\n" + ("\n".join(retrieved_reuters_urls) if retrieved_reuters_urls else "NONE"))
        print("\n\033[92m************************** RUNNING EVALUATION **************************\033[0m")
        news_evaluator_result = await Runner.run(news_evaluator, evaluator_input)
        result: EvaluationFeedback = news_evaluator_result.final_output
        print(f"\033[94mEvaluator score: {result.score}\033[0m")
        print(f"\033[94mEvaluator feedback: {result.feedback}\033[0m")
        if result.score == "successful":
            print("\033[92mEvaluation successful ==> stopping iteration.\033[0m")
            break
        evaluator_feedback = result.feedback
        if iteration == max_iterations: print("\033[91mReached max_iterations ==> stopping iteration.\033[0m")
    print("\n\033[92m************************** FINAL NEWS SET **************************\033[0m")
    print(latest_outline)


## 6. Run

Example: `Give me the latest 5 Reuters articles from the last 2 days about OpenAI, Nvidia, Oracle, Adobe, AI infrastructure or AI investment in the United States.`


In [24]:
await main()


User's request: Give me the latest 5 Reuters articles from the last 2 days about OpenAI, Nvidia, Oracle, Adobe, AI infrastructure or AI investment in the United States.

************************** NEWS SEARCH 1 **************************
I searched for recent Reuters articles from the past two days (September 9–11, 2026) on OpenAI, Nvidia, Oracle, Adobe, AI infrastructure, and AI investment in the United States. Unfortunately, I couldn't find any relevant articles within that timeframe. It's possible that no such articles have been published recently. If you have any other questions or need information on a different topic, feel free to ask. 

************************** WEB SEARCH DIAGNOSTICS **************************
All URLs retrieved by web search: 0
Reuters citations in final response: 0
Reuters URLs retrieved by web search: 0

************************** RUNNING EVALUATION **************************
Evaluator score: unsuccessful
Evaluator feedback: The response correctly shows tha